In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Load CSV
df = pd.read_csv(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\4Pv3.4.1-S_BYBIT_MEMEUSDT.P_2026-02-06_fcc84.csv', encoding='utf-8-sig')

# Parse — each trade has Entry + Exit row, same Trade #
# Keep only Exit rows (they contain the P&L)
exits = df[df['Type'].str.contains('Exit', case=False)].copy()
entries = df[df['Type'].str.contains('Entry', case=False)].copy()

# Build trades dataframe
trades = exits[['Trade #', 'Type', 'Signal', 'Date and time', 'Price USDT', 
                 'Net P&L USDT', 'Net P&L %', 
                 'Favorable excursion USDT', 'Favorable excursion %',
                 'Adverse excursion USDT', 'Adverse excursion %',
                 'Cumulative P&L USDT', 'Cumulative P&L %']].copy()

# Merge entry info
entries_info = entries[['Trade #', 'Date and time', 'Price USDT', 'Type']].copy()
entries_info.columns = ['Trade #', 'Entry Time', 'Entry Price', 'Entry Type']
trades = trades.merge(entries_info, on='Trade #', how='left')
trades.rename(columns={'Date and time': 'Exit Time', 'Price USDT': 'Exit Price', 'Type': 'Exit Type'}, inplace=True)

# Direction from entry type
trades['Direction'] = trades['Entry Type'].apply(lambda x: 'Long' if 'long' in x.lower() else 'Short')

# Apply 20x leverage to P&L
LEVERAGE = 20
trades['PnL_20x'] = trades['Net P&L USDT'] * LEVERAGE
trades['PnL_pct_20x'] = trades['Net P&L %'] * LEVERAGE
trades['CumPnL_20x'] = trades['PnL_20x'].cumsum()
trades['MFE_20x'] = trades['Favorable excursion USDT'] * LEVERAGE
trades['MAE_20x'] = trades['Adverse excursion USDT'] * LEVERAGE

# Win/Loss
trades['Win'] = trades['PnL_20x'] > 0
trades['Loss'] = trades['PnL_20x'] < 0

print(f"Loaded {len(trades)} trades")
print(f"Date range: {trades['Entry Time'].iloc[0]} → {trades['Exit Time'].iloc[-1]}")
trades.head()

In [ ]:
# ═══════════════════════════════════════════════════════
# CONSECUTIVE LOSS / WIN STREAK ANALYSIS
# ═══════════════════════════════════════════════════════

def compute_streaks(series):
    """Compute consecutive streaks of True values."""
    streaks = []
    current = 0
    for val in series:
        if val:
            current += 1
        else:
            if current > 0:
                streaks.append(current)
            current = 0
    if current > 0:
        streaks.append(current)
    return streaks

# Consecutive losses
loss_streaks = compute_streaks(trades['Loss'])
win_streaks = compute_streaks(trades['Win'])

# Also compute running streak at each trade
trades['streak'] = 0
streak = 0
for i in range(len(trades)):
    if trades.iloc[i]['Loss']:
        streak = streak - 1 if streak < 0 else -1
    elif trades.iloc[i]['Win']:
        streak = streak + 1 if streak > 0 else 1
    else:
        streak = 0
    trades.iloc[i, trades.columns.get_loc('streak')] = streak

max_consec_losses = max(loss_streaks) if loss_streaks else 0
max_consec_wins = max(win_streaks) if win_streaks else 0
avg_loss_streak = np.mean(loss_streaks) if loss_streaks else 0
avg_win_streak = np.mean(win_streaks) if win_streaks else 0

# Find WHERE the max loss streak happened
worst_streak_end = trades['streak'].idxmin()
worst_streak_len = abs(trades.loc[worst_streak_end, 'streak'])
worst_streak_start = worst_streak_end - int(worst_streak_len) + 1
worst_streak_trades = trades.iloc[worst_streak_start:worst_streak_end+1]
worst_streak_damage = worst_streak_trades['PnL_20x'].sum()

print("╔══════════════════════════════════════════════════╗")
print("║         CONSECUTIVE STREAK ANALYSIS              ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Max Consecutive Losses:  {max_consec_losses:>4}                    ║")
print(f"║  Max Consecutive Wins:    {max_consec_wins:>4}                    ║")
print(f"║  Avg Loss Streak:         {avg_loss_streak:>6.1f}                  ║")
print(f"║  Avg Win Streak:          {avg_win_streak:>6.1f}                  ║")
print(f"║  Total Loss Streaks:      {len(loss_streaks):>4}                    ║")
print(f"║  Total Win Streaks:       {len(win_streaks):>4}                    ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  WORST STREAK: {int(worst_streak_len)} consecutive losses           ║")
print(f"║  Damage: ${worst_streak_damage:>10.2f} (20x)                ║")
print(f"║  Trades #{trades.iloc[worst_streak_start]['Trade #']:.0f} – #{trades.iloc[worst_streak_end]['Trade #']:.0f}                        ║")
print(f"║  From: {worst_streak_trades['Entry Time'].iloc[0]}              ║")
print(f"║  To:   {worst_streak_trades['Exit Time'].iloc[-1]}              ║")
print("╚══════════════════════════════════════════════════╝")

# Distribution of loss streaks
from collections import Counter
loss_dist = Counter(loss_streaks)
print("\nLoss Streak Distribution:")
for length in sorted(loss_dist.keys()):
    bar = '█' * loss_dist[length]
    print(f"  {length:>2} in a row: {loss_dist[length]:>3}x  {bar}")

In [ ]:
# ═══════════════════════════════════════════════════════
# CORE PERFORMANCE METRICS (20x)
# ═══════════════════════════════════════════════════════

total_trades = len(trades)
winners = trades[trades['Win']]
losers = trades[trades['Loss']]
breakeven = trades[(~trades['Win']) & (~trades['Loss'])]

win_rate = len(winners) / total_trades * 100
loss_rate = len(losers) / total_trades * 100

total_pnl = trades['PnL_20x'].sum()
avg_win = winners['PnL_20x'].mean() if len(winners) > 0 else 0
avg_loss = losers['PnL_20x'].mean() if len(losers) > 0 else 0
largest_win = trades['PnL_20x'].max()
largest_loss = trades['PnL_20x'].min()
median_win = winners['PnL_20x'].median() if len(winners) > 0 else 0
median_loss = losers['PnL_20x'].median() if len(losers) > 0 else 0

# Profit Factor
gross_profit = winners['PnL_20x'].sum() if len(winners) > 0 else 0
gross_loss = abs(losers['PnL_20x'].sum()) if len(losers) > 0 else 1
profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

# Payoff Ratio (avg win / avg loss)
payoff_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else float('inf')

# Expectancy per trade
expectancy = total_pnl / total_trades

# Long vs Short breakdown
longs = trades[trades['Direction'] == 'Long']
shorts = trades[trades['Direction'] == 'Short']
long_wr = longs['Win'].sum() / len(longs) * 100 if len(longs) > 0 else 0
short_wr = shorts['Win'].sum() / len(shorts) * 100 if len(shorts) > 0 else 0
long_pnl = longs['PnL_20x'].sum()
short_pnl = shorts['PnL_20x'].sum()

print("╔══════════════════════════════════════════════════════════════╗")
print("║            FOUR PILLARS v3.4.1 — TRADE DASHBOARD           ║")
print("║              MEMEUSDT.P  •  20x Leverage                   ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Total Trades:        {total_trades:>6}                              ║")
print(f"║  Winners:             {len(winners):>6}  ({win_rate:.1f}%)                     ║")
print(f"║  Losers:              {len(losers):>6}  ({loss_rate:.1f}%)                     ║")
print(f"║  Breakeven:           {len(breakeven):>6}                              ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Net P&L (20x):     ${total_pnl:>10.2f}                         ║")
print(f"║  Gross Profit:      ${gross_profit:>10.2f}                         ║")
print(f"║  Gross Loss:       -${gross_loss:>10.2f}                         ║")
print(f"║  Profit Factor:      {profit_factor:>10.3f}                         ║")
print(f"║  Payoff Ratio:       {payoff_ratio:>10.3f}  (avg W / avg L)        ║")
print(f"║  Expectancy/Trade:  ${expectancy:>10.2f}                         ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Avg Winner:        ${avg_win:>10.2f}   Median: ${median_win:>8.2f}      ║")
print(f"║  Avg Loser:         ${avg_loss:>10.2f}   Median: ${median_loss:>8.2f}      ║")
print(f"║  Largest Win:       ${largest_win:>10.2f}                         ║")
print(f"║  Largest Loss:      ${largest_loss:>10.2f}                         ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  LONG  trades: {len(longs):>5}  WR: {long_wr:.1f}%  P&L: ${long_pnl:>10.2f}   ║")
print(f"║  SHORT trades: {len(shorts):>5}  WR: {short_wr:.1f}%  P&L: ${short_pnl:>10.2f}   ║")
print("╚══════════════════════════════════════════════════════════════╝")

In [ ]:
# ═══════════════════════════════════════════════════════
# DRAWDOWN & RISK METRICS
# ═══════════════════════════════════════════════════════

# Equity curve
equity = trades['CumPnL_20x']
running_max = equity.cummax()
drawdown = equity - running_max
trades['Drawdown_20x'] = drawdown.values

max_dd = drawdown.min()
max_dd_idx = drawdown.idxmin()
max_dd_trade = trades.loc[max_dd_idx, 'Trade #']

# Find peak before max DD
peak_before_dd = running_max.loc[:max_dd_idx].max()

# Recovery — did it recover?
post_dd = equity.loc[max_dd_idx:]
recovered = post_dd[post_dd >= peak_before_dd]
if len(recovered) > 0:
    recovery_trade = trades.loc[recovered.index[0], 'Trade #']
    recovery_trades_needed = recovered.index[0] - max_dd_idx
    recovery_status = f"Recovered at trade #{recovery_trade:.0f} ({recovery_trades_needed} trades later)"
else:
    recovery_status = "NOT RECOVERED"

# Sharpe-like ratio (using trade returns)
if trades['PnL_20x'].std() > 0:
    sharpe_like = trades['PnL_20x'].mean() / trades['PnL_20x'].std()
else:
    sharpe_like = 0

# Sortino-like (downside deviation only)
downside = trades[trades['PnL_20x'] < 0]['PnL_20x']
if len(downside) > 0 and downside.std() > 0:
    sortino_like = trades['PnL_20x'].mean() / downside.std()
else:
    sortino_like = 0

# Calmar-like (return / max DD)
calmar_like = total_pnl / abs(max_dd) if max_dd != 0 else 0

# Recovery Factor
recovery_factor = total_pnl / abs(max_dd) if max_dd != 0 else 0

# Win/Loss ratio by exit type
exit_types = trades.groupby('Signal').agg(
    count=('PnL_20x', 'count'),
    total_pnl=('PnL_20x', 'sum'),
    avg_pnl=('PnL_20x', 'mean'),
    win_rate=('Win', 'mean')
).round(2)
exit_types['win_rate'] = (exit_types['win_rate'] * 100).round(1)

print("╔══════════════════════════════════════════════════════════════╗")
print("║                 RISK MANAGEMENT METRICS                     ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Max Drawdown (20x):  ${max_dd:>10.2f}  (at trade #{max_dd_trade:.0f})     ║")
print(f"║  Peak Before DD:      ${peak_before_dd:>10.2f}                        ║")
print(f"║  Recovery:  {recovery_status:<46}  ║")
print(f"║  Recovery Factor:     {recovery_factor:>10.3f}  (net P&L / max DD)    ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Sharpe-like:         {sharpe_like:>10.4f}  (mean/std per trade)   ║")
print(f"║  Sortino-like:        {sortino_like:>10.4f}  (mean/downside std)   ║")
print(f"║  Calmar-like:         {calmar_like:>10.4f}  (net P&L / max DD)     ║")
print("╚══════════════════════════════════════════════════════════════╝")

print("\n═══ Exit Signal Breakdown ═══")
print(exit_types.to_string())

In [ ]:
# ═══════════════════════════════════════════════════════
# VISUAL DASHBOARD
# ═══════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 20), facecolor='#1a1a2e')
gs = GridSpec(4, 2, figure=fig, hspace=0.35, wspace=0.3)

# Style helper
def style_ax(ax, title):
    ax.set_facecolor('#16213e')
    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=10)
    ax.tick_params(colors='#aaa')
    for spine in ax.spines.values():
        spine.set_color('#333')
    ax.grid(True, alpha=0.15, color='white')

# 1. Equity Curve (20x)
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(range(len(equity)), equity, alpha=0.3, 
                 color=np.where(equity >= 0, '#00d4aa', '#ff4757')[0])
ax1.plot(equity.values, color='#00d4aa', linewidth=1.2)
ax1.axhline(y=0, color='#666', linestyle='--', linewidth=0.8)
style_ax(ax1, 'EQUITY CURVE (20x Leverage)')
ax1.set_xlabel('Trade #', color='#aaa')
ax1.set_ylabel('Cumulative P&L ($)', color='#aaa')
# Mark max drawdown point
ax1.scatter([max_dd_idx], [equity.iloc[max_dd_idx]], color='#ff4757', s=100, zorder=5, label=f'Max DD: ${max_dd:.0f}')
ax1.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white')

# 2. Drawdown Chart
ax2 = fig.add_subplot(gs[1, :])
ax2.fill_between(range(len(drawdown)), drawdown, alpha=0.5, color='#ff4757')
ax2.plot(drawdown.values, color='#ff6b81', linewidth=0.8)
style_ax(ax2, 'DRAWDOWN (20x)')
ax2.set_xlabel('Trade #', color='#aaa')
ax2.set_ylabel('Drawdown ($)', color='#aaa')

# 3. Trade P&L Distribution
ax3 = fig.add_subplot(gs[2, 0])
colors = ['#00d4aa' if x > 0 else '#ff4757' for x in trades['PnL_20x']]
ax3.bar(range(len(trades)), trades['PnL_20x'], color=colors, width=1.0, alpha=0.7)
style_ax(ax3, 'P&L PER TRADE (20x)')
ax3.set_xlabel('Trade #', color='#aaa')
ax3.set_ylabel('P&L ($)', color='#aaa')

# 4. P&L Histogram
ax4 = fig.add_subplot(gs[2, 1])
ax4.hist(trades['PnL_20x'], bins=50, color='#5dade2', alpha=0.7, edgecolor='#333')
ax4.axvline(x=0, color='white', linestyle='--', linewidth=0.8)
ax4.axvline(x=trades['PnL_20x'].mean(), color='#ffd32a', linestyle='-', linewidth=1.5, label=f'Mean: ${expectancy:.2f}')
ax4.axvline(x=trades['PnL_20x'].median(), color='#ff4757', linestyle='-', linewidth=1.5, label=f'Median: ${trades["PnL_20x"].median():.2f}')
style_ax(ax4, 'P&L DISTRIBUTION (20x)')
ax4.set_xlabel('P&L ($)', color='#aaa')
ax4.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=9)

# 5. Streak chart
ax5 = fig.add_subplot(gs[3, 0])
streak_colors = ['#00d4aa' if s > 0 else '#ff4757' if s < 0 else '#666' for s in trades['streak']]
ax5.bar(range(len(trades)), trades['streak'], color=streak_colors, width=1.0, alpha=0.7)
style_ax(ax5, 'WIN/LOSS STREAKS')
ax5.set_xlabel('Trade #', color='#aaa')
ax5.set_ylabel('Streak Length', color='#aaa')
ax5.axhline(y=0, color='#666', linewidth=0.5)

# 6. Win rate by exit signal (pie)
ax6 = fig.add_subplot(gs[3, 1])
signal_counts = trades['Signal'].value_counts()
colors_pie = ['#00d4aa', '#ff4757', '#5dade2', '#ffd32a', '#c39bd3', '#f0b27a']
wedges, texts, autotexts = ax6.pie(signal_counts.values, labels=signal_counts.index, 
                                    autopct='%1.1f%%', colors=colors_pie[:len(signal_counts)],
                                    textprops={'color': 'white', 'fontsize': 9})
style_ax(ax6, 'EXIT SIGNAL DISTRIBUTION')
ax6.grid(False)

fig.suptitle('FOUR PILLARS v3.4.1 — MEMEUSDT TRADE DASHBOARD', 
             color='white', fontsize=16, fontweight='bold', y=0.98)

plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\trade_dashboard.png', 
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("Dashboard saved to trade_dashboard.png")

In [ ]:
# ═══════════════════════════════════════════════════════
# TIME & DURATION ANALYSIS
# ═══════════════════════════════════════════════════════

trades['Entry_dt'] = pd.to_datetime(trades['Entry Time'])
trades['Exit_dt'] = pd.to_datetime(trades['Exit Time'])
trades['Duration'] = trades['Exit_dt'] - trades['Entry_dt']
trades['Duration_mins'] = trades['Duration'].dt.total_seconds() / 60

# Hour of day analysis
trades['Entry_hour'] = trades['Entry_dt'].dt.hour
hourly = trades.groupby('Entry_hour').agg(
    count=('PnL_20x', 'count'),
    total_pnl=('PnL_20x', 'sum'),
    avg_pnl=('PnL_20x', 'mean'),
    win_rate=('Win', 'mean')
).round(2)
hourly['win_rate'] = (hourly['win_rate'] * 100).round(1)

# Day of week
trades['DOW'] = trades['Entry_dt'].dt.day_name()
daily = trades.groupby('DOW').agg(
    count=('PnL_20x', 'count'),
    total_pnl=('PnL_20x', 'sum'),
    avg_pnl=('PnL_20x', 'mean'),
    win_rate=('Win', 'mean')
).round(2)
daily['win_rate'] = (daily['win_rate'] * 100).round(1)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = daily.reindex([d for d in day_order if d in daily.index])

print("╔══════════════════════════════════════════════════════════════╗")
print("║                   TIME ANALYSIS                            ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Avg Trade Duration:  {trades['Duration_mins'].mean():>8.1f} min                     ║")
print(f"║  Median Duration:     {trades['Duration_mins'].median():>8.1f} min                     ║")
print(f"║  Shortest Trade:      {trades['Duration_mins'].min():>8.1f} min                     ║")
print(f"║  Longest Trade:       {trades['Duration_mins'].max():>8.1f} min                     ║")
print("╚══════════════════════════════════════════════════════════════╝")

# Winner vs Loser duration
w_dur = winners['Duration_mins'].mean() if 'Duration_mins' in winners.columns else trades.loc[winners.index, 'Duration_mins'].mean()
l_dur = losers['Duration_mins'].mean() if 'Duration_mins' in losers.columns else trades.loc[losers.index, 'Duration_mins'].mean()
print(f"\nAvg Winner Duration: {w_dur:.1f} min")
print(f"Avg Loser Duration:  {l_dur:.1f} min")

print("\n═══ P&L by Hour (UTC) ═══")
print(hourly.to_string())

print("\n═══ P&L by Day of Week ═══")
print(daily.to_string())

In [ ]:
# ═══════════════════════════════════════════════════════
# MFE / MAE ANALYSIS (20x)
# ═══════════════════════════════════════════════════════

fig2, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#1a1a2e')

# MFE vs Actual P&L
ax = axes[0]
ax.set_facecolor('#16213e')
win_mask = trades['Win']
ax.scatter(trades.loc[win_mask, 'MFE_20x'], trades.loc[win_mask, 'PnL_20x'], 
           c='#00d4aa', alpha=0.4, s=15, label='Winners')
ax.scatter(trades.loc[~win_mask, 'MFE_20x'], trades.loc[~win_mask, 'PnL_20x'], 
           c='#ff4757', alpha=0.4, s=15, label='Losers')
ax.axhline(y=0, color='#666', linestyle='--', linewidth=0.8)
# Diagonal = captured all favorable excursion
max_mfe = trades['MFE_20x'].max()
ax.plot([0, max_mfe], [0, max_mfe], color='#ffd32a', linestyle=':', alpha=0.5, label='100% capture')
ax.set_title('MFE vs Actual P&L (20x)', color='white', fontweight='bold')
ax.set_xlabel('Max Favorable Excursion ($)', color='#aaa')
ax.set_ylabel('Actual P&L ($)', color='#aaa')
ax.tick_params(colors='#aaa')
ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=8)
ax.grid(True, alpha=0.15)
for spine in ax.spines.values(): spine.set_color('#333')

# MAE vs Actual P&L  
ax = axes[1]
ax.set_facecolor('#16213e')
ax.scatter(trades.loc[win_mask, 'MAE_20x'], trades.loc[win_mask, 'PnL_20x'], 
           c='#00d4aa', alpha=0.4, s=15, label='Winners')
ax.scatter(trades.loc[~win_mask, 'MAE_20x'], trades.loc[~win_mask, 'PnL_20x'], 
           c='#ff4757', alpha=0.4, s=15, label='Losers')
ax.axhline(y=0, color='#666', linestyle='--', linewidth=0.8)
ax.set_title('MAE vs Actual P&L (20x)', color='white', fontweight='bold')
ax.set_xlabel('Max Adverse Excursion ($)', color='#aaa')
ax.set_ylabel('Actual P&L ($)', color='#aaa')
ax.tick_params(colors='#aaa')
ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white', fontsize=8)
ax.grid(True, alpha=0.15)
for spine in ax.spines.values(): spine.set_color('#333')

fig2.suptitle('TRADE EFFICIENCY — MFE / MAE ANALYSIS', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\mfe_mae_analysis.png', 
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

# Capture efficiency
if len(winners) > 0:
    capture_eff = (winners['PnL_20x'] / winners['MFE_20x'].replace(0, np.nan)).dropna().mean() * 100
    print(f"\nAvg MFE Capture Efficiency (winners): {capture_eff:.1f}%")
    print(f"  → Winners captured {capture_eff:.1f}% of their max favorable move on average")

# How many losers were in profit at some point?
losers_had_profit = (losers['MFE_20x'] > 0).sum()
print(f"\nLosers that were profitable at some point: {losers_had_profit}/{len(losers)} ({losers_had_profit/len(losers)*100:.1f}%)")
if losers_had_profit > 0:
    avg_missed = losers[losers['MFE_20x'] > 0]['MFE_20x'].mean()
    print(f"  → Avg missed profit on those losers: ${avg_missed:.2f}")

In [ ]:
# ═══════════════════════════════════════════════════════
# ROLLING PERFORMANCE (50-trade window)
# ═══════════════════════════════════════════════════════

window = 50
trades['Rolling_WR'] = trades['Win'].rolling(window).mean() * 100
trades['Rolling_PnL'] = trades['PnL_20x'].rolling(window).sum()
trades['Rolling_Avg'] = trades['PnL_20x'].rolling(window).mean()

fig3, axes = plt.subplots(3, 1, figsize=(16, 12), facecolor='#1a1a2e')

# Rolling win rate
ax = axes[0]
ax.set_facecolor('#16213e')
ax.plot(trades['Rolling_WR'], color='#5dade2', linewidth=1.2)
ax.axhline(y=50, color='#ff4757', linestyle='--', linewidth=0.8, label='50% WR')
ax.set_title(f'Rolling {window}-Trade Win Rate', color='white', fontweight='bold')
ax.set_ylabel('Win Rate %', color='#aaa')
ax.tick_params(colors='#aaa')
ax.legend(facecolor='#16213e', edgecolor='#333', labelcolor='white')
ax.grid(True, alpha=0.15)
for spine in ax.spines.values(): spine.set_color('#333')

# Rolling P&L sum
ax = axes[1]
ax.set_facecolor('#16213e')
ax.fill_between(range(len(trades)), trades['Rolling_PnL'].fillna(0), alpha=0.3,
                color='#00d4aa')
ax.plot(trades['Rolling_PnL'], color='#00d4aa', linewidth=1.0)
ax.axhline(y=0, color='#ff4757', linestyle='--', linewidth=0.8)
ax.set_title(f'Rolling {window}-Trade P&L Sum (20x)', color='white', fontweight='bold')
ax.set_ylabel('P&L ($)', color='#aaa')
ax.tick_params(colors='#aaa')
ax.grid(True, alpha=0.15)
for spine in ax.spines.values(): spine.set_color('#333')

# Rolling average per trade
ax = axes[2]
ax.set_facecolor('#16213e')
ax.plot(trades['Rolling_Avg'], color='#ffd32a', linewidth=1.0)
ax.axhline(y=0, color='#ff4757', linestyle='--', linewidth=0.8)
ax.set_title(f'Rolling {window}-Trade Avg P&L (20x)', color='white', fontweight='bold')
ax.set_xlabel('Trade #', color='#aaa')
ax.set_ylabel('Avg P&L ($)', color='#aaa')
ax.tick_params(colors='#aaa')
ax.grid(True, alpha=0.15)
for spine in ax.spines.values(): spine.set_color('#333')

fig3.suptitle('ROLLING PERFORMANCE METRICS', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Users\User\Documents\Obsidian Vault\07-TEMPLATES\rolling_metrics.png', 
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# TOP 10 BEST / WORST TRADES
# ═══════════════════════════════════════════════════════

print("═══ TOP 10 BEST TRADES (20x) ═══")
best = trades.nlargest(10, 'PnL_20x')[['Trade #', 'Direction', 'Entry Time', 'Exit Time', 'Signal', 'PnL_20x', 'Duration_mins']]
best['PnL_20x'] = best['PnL_20x'].apply(lambda x: f'${x:.2f}')
best['Duration_mins'] = best['Duration_mins'].apply(lambda x: f'{x:.0f}m')
print(best.to_string(index=False))

print("\n═══ TOP 10 WORST TRADES (20x) ═══")
worst = trades.nsmallest(10, 'PnL_20x')[['Trade #', 'Direction', 'Entry Time', 'Exit Time', 'Signal', 'PnL_20x', 'Duration_mins']]
worst['PnL_20x'] = worst['PnL_20x'].apply(lambda x: f'${x:.2f}')
worst['Duration_mins'] = worst['Duration_mins'].apply(lambda x: f'{x:.0f}m')
print(worst.to_string(index=False))

# Quick summary
print("\n" + "="*60)
print("EXECUTIVE SUMMARY")
print("="*60)
print(f"  {total_trades} trades over {(trades['Exit_dt'].iloc[-1] - trades['Entry_dt'].iloc[0]).days} days")
print(f"  Net P&L at 20x: ${total_pnl:.2f}")
print(f"  Win Rate: {win_rate:.1f}% | Profit Factor: {profit_factor:.3f}")
print(f"  Expectancy: ${expectancy:.2f}/trade")
print(f"  Max Drawdown: ${max_dd:.2f}")
print(f"  Max Consecutive Losses: {max_consec_losses}")
print(f"  Worst streak damage: ${worst_streak_damage:.2f}")
verdict = "PROFITABLE" if total_pnl > 0 else "UNPROFITABLE"
print(f"\n  VERDICT: {verdict}")